# EDA de 52 preguntas — Importaciones Aduana de Buenaventura

**Producto de datos · ADUA 35 · Universidad Libre, Seccional Cali**

Juan Manuel Tejada Fajardo · Jesús Alejandro Guerrero

---

### Cómo está organizado

Cada pregunta sigue siempre la misma estructura:

| | |
|---|---|
| **Pregunta** | El enunciado, con su método y la representación esperada |
| **Código** | La celda que calcula. Es visible a propósito: la respuesta debe poder auditarse |
| **Gráfico** | La evidencia visual |
| **Respuesta** | Qué muestran los datos, en números |
| **Explicación** | Qué significa, qué implica para el modelo y qué limitación tiene |

### Qué se puede ejecutar aquí

Este notebook trabaja con los datos **ya consolidados** que viven en el repositorio: la serie mensual de 173 meses, los agregados por país y capítulo, las variables externas y los resultados del backtest. No necesita los 8,3 GB de microdatos del DANE.

Eso permite responder las preguntas de cobertura, calidad agregada, distribución, valor unitario, dinámica temporal, composición, variables externas y modelado.

Las preguntas que operan sobre el registro individual —esquemas por vigencia, duplicados por capa, ceros iniciales en los códigos, reconciliación contra el boletín oficial— **requieren los microdatos completos** y deben ejecutarse en la máquina local. Están señaladas al final con lo que hace falta para cada una.

> Esta separación es deliberada. Es preferible decir qué falta a fabricar una respuesta.

## Preparación

In [ ]:
# Descarga el repositorio. Si ya lo clonaste antes, esta celda no hace nada.
import os

REPO = 'https://github.com/SoftEngineer96/Proyecto-Buenaventura.git'

if not os.path.exists('Proyecto-Buenaventura'):
    !git clone -q $REPO
    print('Repositorio clonado.')
else:
    print('El repositorio ya estaba clonado.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, HTML

# --- Localiza la raíz del proyecto funcione donde funcione ---
CANDIDATOS = [Path('Proyecto-Buenaventura'), Path('.'), Path('..')]
ROOT = next((c.resolve() for c in CANDIDATOS if (c / 'data' / 'trusted').exists()), None)
if ROOT is None:
    raise FileNotFoundError('No encuentro la carpeta data/trusted. Ejecuta primero la celda de clonado.')

TRUSTED, SURFACE = ROOT / 'data' / 'trusted', ROOT / 'data' / 'surface'
EXTERNAL, REPORTS = ROOT / 'data' / 'external', ROOT / 'reportes'
print('Raíz del proyecto:', ROOT)

# --- Estilo de las figuras ---
plt.rcParams.update({
    'figure.figsize': (11, 4.2), 'figure.dpi': 110,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'font.size': 10,
})
AZUL, NARANJA, VERDE, ROJO, GRIS = '#1f3864', '#c8791a', '#0f7b4f', '#a12b2b', '#8894a6'


def millones(eje, div=1e6, sufijo=' M'):
    eje.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/div:,.0f}{sufijo}'))


# --- Bloques de presentación ---
def pregunta(pid, texto, metodo, representacion, uso):
    display(HTML(f"""
    <div style="border-left:5px solid #1f3864;background:#f4f7fc;padding:14px 18px;
                margin:26px 0 10px;border-radius:0 8px 8px 0;font-family:system-ui,sans-serif">
      <div style="font-size:12px;letter-spacing:1.2px;color:#1f3864;font-weight:700">{pid}</div>
      <div style="font-size:17px;font-weight:600;color:#16233a;margin:4px 0 10px">{texto}</div>
      <div style="font-size:13px;color:#4a5a6d;line-height:1.6">
        <b>Método:</b> {metodo}<br>
        <b>Representación:</b> {representacion}<br>
        <b>Uso en el producto:</b> {uso}
      </div>
    </div>"""))


def respuesta(hallazgo, interpretacion, implicacion, limitacion):
    display(HTML(f"""
    <div style="border:1px solid #cfe0d6;background:#f2faf5;padding:14px 18px;
                margin:8px 0 30px;border-radius:8px;font-family:system-ui,sans-serif;font-size:14px;line-height:1.65">
      <div style="color:#0f7b4f;font-weight:700;font-size:12px;letter-spacing:1.2px;margin-bottom:8px">RESPUESTA</div>
      <p style="margin:0 0 10px"><b>Evidencia.</b> {hallazgo}</p>
      <p style="margin:0 0 10px"><b>Interpretación.</b> {interpretacion}</p>
      <p style="margin:0 0 10px"><b>Implicación.</b> {implicacion}</p>
      <p style="margin:0;color:#7a6a4a"><b>Limitación.</b> {limitacion}</p>
    </div>"""))


def aviso(texto):
    display(HTML(f"""<div style="border-left:4px solid #c8791a;background:#fff8ec;padding:12px 16px;
      margin:10px 0;border-radius:0 6px 6px 0;font-family:system-ui,sans-serif;font-size:14px">{texto}</div>"""))

print('Entorno listo.')

In [ ]:
# Carga de datos y diagnóstico de lo que hay disponible
serie = pd.read_csv(TRUSTED / 'serie_mensual_buenaventura.csv', parse_dates=['fecha']).sort_values('fecha')
serie = serie.set_index('fecha')

ext = pd.read_csv(EXTERNAL / 'variables_externas_mensuales.csv', parse_dates=['fecha']).set_index('fecha').sort_index()

def carga_opcional(ruta, **kw):
    try:
        return pd.read_csv(ruta, **kw)
    except Exception as e:
        print(f'  no disponible: {ruta.name} ({type(e).__name__})')
        return None

pais = carga_opcional(TRUSTED / 'pais_mes_buenaventura.csv.gz')
capitulo = carga_opcional(TRUSTED / 'capitulo_mes_buenaventura.csv.gz')
preds = carga_opcional(SURFACE / 'predicciones_validacion.csv', parse_dates=['fecha'])
metricas = carga_opcional(REPORTS / 'metricas_modelos.csv')
eda_hist = json.loads((TRUSTED / 'eda_results.json').read_text(encoding='utf-8'))

print('\nSerie mensual:', serie.shape, '|', serie.index.min().date(), 'a', serie.index.max().date())
print('Columnas:', list(serie.columns))
print('\nExternas:', list(ext.columns))
for nombre, df in [('País-mes', pais), ('Capítulo-mes', capitulo)]:
    if df is not None:
        print(f'{nombre}: {df.shape} | columnas: {list(df.columns)}')

---
# Bloque 1 · Fuentes, estructura y cobertura

In [ ]:
pregunta('P05',
  '¿La cobertura mensual es continua y cuáles periodos faltan, si existen?',
  'Calendario mensual esperado contra observado.',
  'Línea de tiempo con los meses faltantes marcados.',
  'Validar que la serie no tenga huecos antes de modelar.')

esperado = pd.date_range(serie.index.min(), serie.index.max(), freq='MS')
faltantes = esperado.difference(serie.index)
duplicados = serie.index.duplicated().sum()

fig, ax = plt.subplots(figsize=(11, 2.4))
ax.scatter(serie.index, np.ones(len(serie)), s=14, color=VERDE, label=f'{len(serie)} meses presentes')
if len(faltantes):
    ax.scatter(faltantes, np.ones(len(faltantes)), s=60, color=ROJO, marker='x', label='faltante')
ax.set_yticks([]); ax.legend(loc='upper left', frameon=False)
ax.set_title('P05 · Cobertura mensual de la serie')
plt.tight_layout(); plt.show()

respuesta(
  f'La serie va de <b>{serie.index.min():%B de %Y}</b> a <b>{serie.index.max():%B de %Y}</b>: '
  f'<b>{len(esperado)} meses esperados</b> y <b>{len(serie)} observados</b>. '
  f'Meses faltantes: <b>{len(faltantes)}</b>. Fechas duplicadas: <b>{duplicados}</b>.',
  'La cobertura temporal es continua y sin repeticiones. Cada mes del calendario aparece exactamente una vez.',
  'Se puede tratar como una serie de frecuencia mensual regular, sin imputación de huecos, y aplicar '
  'rezagos y medias móviles sin discontinuidades que distorsionen los cálculos.',
  'La continuidad confirma que no falta ningún mes, pero no dice nada sobre si el contenido de cada mes '
  'es correcto. Eso depende de la reconciliación contra los totales oficiales del DANE, que exige los microdatos.')

In [ ]:
pregunta('P09',
  '¿Cuál es la completitud de las variables de la serie y de las externas por periodo?',
  'Conteo de valores nulos por variable y por año.',
  'Mapa de calor de completitud.',
  'Definir qué variables son utilizables como objetivo o como predictor.')

combinado = serie.join(ext, how='left')
nulos_anio = combinado.isna().groupby(combinado.index.year).mean() * 100

fig, ax = plt.subplots(figsize=(11, max(3.2, 0.30 * len(nulos_anio.columns))))
im = ax.imshow(nulos_anio.T.values, aspect='auto', cmap='Reds', vmin=0, vmax=100)
ax.set_yticks(range(len(nulos_anio.columns))); ax.set_yticklabels(nulos_anio.columns, fontsize=8)
ax.set_xticks(range(len(nulos_anio.index))); ax.set_xticklabels(nulos_anio.index, rotation=90, fontsize=8)
ax.set_title('P09 · Porcentaje de valores faltantes por variable y año'); ax.grid(False)
plt.colorbar(im, ax=ax, label='% nulos'); plt.tight_layout(); plt.show()

total_nulos = combinado.isna().sum()
incompletas = total_nulos[total_nulos > 0]

respuesta(
  ('Ninguna variable presenta valores faltantes en todo el periodo.' if incompletas.empty else
   'Variables con faltantes: <b>' + ', '.join(f'{k} ({v})' for k, v in incompletas.items()) + '</b>.'),
  ('Las series objetivo y las variables externas están completas, así que no hace falta imputar.'
   if incompletas.empty else
   'Hay huecos que deben resolverse antes de modelar. Si se concentran en los primeros o últimos meses, '
   'suelen deberse al desfase de publicación de la fuente externa, no a un error.'),
  'Los objetivos CIF y peso neto pueden usarse directamente. Cualquier imputación que se decida debe '
  'ajustarse dentro de cada corte de validación, nunca sobre la serie completa, para no filtrar información.',
  'La completitud del agregado mensual no garantiza la del registro individual: un mes puede estar completo '
  'y aun así haberse construido con registros que traían campos vacíos.')

In [ ]:
pregunta('P11',
  '¿Qué coherencia existe entre el valor CIF y el valor FOB?',
  'Razón CIF/FOB y diferencia absoluta mes a mes.',
  'Serie de la razón y dispersión CIF contra FOB.',
  'Detectar inconsistencias de captura en las variables monetarias.')

razon = serie['cif_usd'] / serie['fob_usd']
violaciones = int((serie['cif_usd'] < serie['fob_usd']).sum())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4))
a1.plot(razon.index, razon.values, color=AZUL, lw=1.5)
a1.axhline(1.0, color=ROJO, ls='--', lw=1, label='CIF = FOB (mínimo teórico)')
a1.set_title('Razón CIF / FOB'); a1.legend(frameon=False, fontsize=9)

a2.scatter(serie['fob_usd'] / 1e6, serie['cif_usd'] / 1e6, s=18, color=AZUL, alpha=.65)
lim = [0, max(serie['cif_usd'].max(), serie['fob_usd'].max()) / 1e6 * 1.05]
a2.plot(lim, lim, color=ROJO, ls='--', lw=1)
a2.set_xlabel('FOB (millones USD)'); a2.set_ylabel('CIF (millones USD)')
a2.set_title('CIF contra FOB')
plt.suptitle('P11 · Coherencia entre valor CIF y valor FOB', y=1.02, fontweight='bold')
plt.tight_layout(); plt.show()

sobrecosto = (razon - 1) * 100
respuesta(
  f'La razón CIF/FOB tiene mediana <b>{razon.median():.4f}</b> y se mueve entre '
  f'<b>{razon.min():.4f}</b> y <b>{razon.max():.4f}</b>. Meses en que el CIF resulta menor que el FOB: '
  f'<b>{violaciones}</b>. El sobrecosto de seguro y flete promedia <b>{sobrecosto.mean():.2f} %</b> del FOB.',
  'El CIF incorpora mercancía más seguro y flete, así que por definición debe ser mayor o igual que el FOB. '
  'La regla se cumple y el margen se mantiene en un rango estrecho y estable.',
  'Las dos variables monetarias son mutuamente consistentes. La razón CIF/FOB sirve además como indicador '
  'del costo logístico agregado y puede seguirse en el panel descriptivo.',
  'La coherencia se verifica sobre el agregado mensual. Un registro individual incoherente puede quedar '
  'compensado por otros dentro del mismo mes y no aparecer aquí.')

---
# Bloque 3 · Distribución, valor unitario y extremos

In [ ]:
pregunta('P17 · P18',
  '¿Cómo se distribuyen el valor CIF y el peso neto mensual?',
  'Resumen de percentiles, asimetría y curtosis.',
  'Histograma y diagrama de caja por indicador.',
  'Comprender la forma de la distribución antes de elegir modelo y transformación.')

fig, ejes = plt.subplots(2, 2, figsize=(12.5, 7))
for fila, (col, etiqueta, color) in enumerate([
        ('cif_usd', 'Valor CIF (millones USD)', AZUL),
        ('peso_neto_kg', 'Peso neto (millones kg)', NARANJA)]):
    v = serie[col] / 1e6
    ejes[fila, 0].hist(v, bins=28, color=color, alpha=.8, edgecolor='white')
    ejes[fila, 0].axvline(v.mean(), color=ROJO, ls='--', lw=1.2, label=f'media {v.mean():,.0f}')
    ejes[fila, 0].axvline(v.median(), color=VERDE, ls='-', lw=1.2, label=f'mediana {v.median():,.0f}')
    ejes[fila, 0].legend(frameon=False, fontsize=9); ejes[fila, 0].set_xlabel(etiqueta)
    ejes[fila, 1].boxplot(v, vert=False, widths=.5,
                          patch_artist=True, boxprops=dict(facecolor=color, alpha=.55))
    ejes[fila, 1].set_yticks([]); ejes[fila, 1].set_xlabel(etiqueta)
plt.suptitle('P17 y P18 · Distribución mensual de los dos objetivos', y=1.0, fontweight='bold')
plt.tight_layout(); plt.show()

resumen = serie[['cif_usd', 'peso_neto_kg', 'precio_implicito_usd_kg']].describe().T
resumen['asimetria'] = serie[resumen.index].skew()
resumen['curtosis'] = serie[resumen.index].kurt()
display(resumen[['mean', 'std', 'min', '50%', 'max', 'asimetria', 'curtosis']].round(3))

sk_cif, sk_peso = serie['cif_usd'].skew(), serie['peso_neto_kg'].skew()
respuesta(
  f'El CIF mensual tiene media <b>{serie["cif_usd"].mean()/1e6:,.0f} M USD</b>, mediana '
  f'<b>{serie["cif_usd"].median()/1e6:,.0f} M</b> y asimetría <b>{sk_cif:.3f}</b>. '
  f'El peso neto tiene media <b>{serie["peso_neto_kg"].mean()/1e6:,.0f} M kg</b> y asimetría <b>{sk_peso:.3f}</b>.',
  'Ambas distribuciones tienen la media por encima de la mediana y cola derecha: hay meses excepcionalmente '
  'altos que estiran el promedio. Es el comportamiento habitual de series de comercio exterior, donde unas '
  'pocas operaciones grandes pesan mucho.',
  'Conviene modelar en escala logarítmica para estabilizar la varianza. El pipeline actual ya lo hace mediante '
  'TransformedTargetRegressor con log1p, lo cual es coherente con esta evidencia.',
  'Es la distribución del agregado mensual, no la del registro individual. La asimetría a nivel de registro '
  'es mucho más extrema y solo puede medirse con los microdatos completos.')

In [ ]:
pregunta('P19 · P26',
  '¿Cómo se distribuye y cómo evoluciona el valor CIF por kilogramo?',
  'Cociente entre valor CIF y peso neto, con control de división por cero.',
  'Serie temporal con media móvil e histograma del valor unitario.',
  'Separar la variación de valor de la variación de cantidad: distinguir si entró más carga o si la carga se encareció.')

if 'precio_implicito_usd_kg' in serie.columns:
    cif_kg = serie['precio_implicito_usd_kg']
else:
    cif_kg = serie['cif_usd'] / serie['peso_neto_kg'].replace(0, np.nan)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4), gridspec_kw={'width_ratios': [2, 1]})
a1.plot(cif_kg.index, cif_kg.values, color=GRIS, lw=1, alpha=.8, label='mensual')
a1.plot(cif_kg.index, cif_kg.rolling(12).mean(), color=ROJO, lw=2.2, label='media móvil 12 meses')
a1.set_ylabel('USD por kg'); a1.legend(frameon=False, fontsize=9); a1.set_title('Evolución del CIF por kilogramo')
a2.hist(cif_kg.dropna(), bins=26, color=VERDE, alpha=.8, edgecolor='white')
a2.set_xlabel('USD por kg'); a2.set_title('Distribución')
plt.suptitle('P19 y P26 · Valor CIF unitario implícito', y=1.02, fontweight='bold')
plt.tight_layout(); plt.show()

ini, fin = cif_kg.iloc[:24].mean(), cif_kg.iloc[-24:].mean()
cambio = (fin / ini - 1) * 100

respuesta(
  f'El CIF por kilogramo va de <b>{cif_kg.min():.3f}</b> a <b>{cif_kg.max():.3f}</b> USD/kg, con mediana '
  f'<b>{cif_kg.median():.3f}</b>. Comparando los primeros 24 meses con los últimos 24, el valor unitario '
  f'pasó de <b>{ini:.3f}</b> a <b>{fin:.3f}</b> USD/kg, un cambio de <b>{cambio:+.1f} %</b>.',
  'Este indicador responde la pregunta que ni el CIF ni el peso responden por separado. Si el CIF sube y el '
  'valor unitario también, el aumento vino sobre todo de precios o de un cambio de mezcla hacia mercancía más '
  'cara. Si el CIF sube y el valor unitario se mantiene, entró más carga.',
  'Debe incorporarse al panel descriptivo junto al CIF y al peso. Es el indicador que evita interpretar un '
  'salto de valor como un salto de volumen, que es el error de lectura más costoso del producto.',
  'Es un valor unitario implícito agregado, <b>no un precio</b>. Mezcla productos muy distintos y está afectado '
  'por seguro, flete y por la composición de la canasta importada. No debe presentarse como un índice de precios.')

In [ ]:
pregunta('P21 · P22',
  '¿Qué meses presentan valores extremos, y coinciden entre indicadores?',
  'Rango intercuartílico y z robusto basado en la desviación absoluta mediana (MAD).',
  'Series con marcadores de extremos y gráfico de cuadrantes valor contra volumen.',
  'Distinguir un error de dato de un evento económico real, y saber si el mes atípico fue de precio o de cantidad.')

def z_robusto(s):
    mad = np.median(np.abs(s - s.median()))
    return (s - s.median()) / (1.4826 * mad) if mad else pd.Series(0.0, index=s.index)

z_cif, z_peso = z_robusto(serie['cif_usd']), z_robusto(serie['peso_neto_kg'])
UMBRAL = 3.0
ext_cif, ext_peso = serie.index[z_cif.abs() > UMBRAL], serie.index[z_peso.abs() > UMBRAL]

fig, ejes = plt.subplots(2, 1, figsize=(11.5, 6), sharex=True)
for ax, col, marc, etiq, color in [
        (ejes[0], 'cif_usd', ext_cif, 'CIF (millones USD)', AZUL),
        (ejes[1], 'peso_neto_kg', ext_peso, 'Peso neto (millones kg)', NARANJA)]:
    ax.plot(serie.index, serie[col], color=color, lw=1.4)
    ax.scatter(marc, serie.loc[marc, col], color=ROJO, s=55, zorder=5, label=f'{len(marc)} extremos')
    millones(ax); ax.set_ylabel(etiq); ax.legend(frameon=False, fontsize=9)
ejes[0].set_title('P21 · Meses con valores extremos (z robusto > 3)')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.scatter(z_cif, z_peso, s=26, color=GRIS, alpha=.7)
coinc = serie.index.intersection(ext_cif).intersection(ext_peso)
if len(coinc):
    ax.scatter(z_cif[coinc], z_peso[coinc], s=70, color=ROJO, label='extremo en ambos')
    ax.legend(frameon=False, fontsize=9)
for v in (-UMBRAL, UMBRAL):
    ax.axhline(v, color=ROJO, ls=':', lw=.9); ax.axvline(v, color=ROJO, ls=':', lw=.9)
ax.axhline(0, color=GRIS, lw=.7); ax.axvline(0, color=GRIS, lw=.7)
ax.set_xlabel('z robusto · CIF'); ax.set_ylabel('z robusto · peso neto')
ax.set_title('P22 · ¿Valor o volumen?')
plt.tight_layout(); plt.show()

solo_valor = ext_cif.difference(ext_peso)
respuesta(
  f'Con umbral de z robusto mayor que 3 se detectan <b>{len(ext_cif)}</b> meses extremos en CIF y '
  f'<b>{len(ext_peso)}</b> en peso neto. Coinciden en <b>{len(coinc)}</b> meses. '
  + (f'Meses extremos solo en valor: <b>{", ".join(d.strftime("%Y-%m") for d in solo_valor[:6])}</b>.' if len(solo_valor) else ''),
  'Cuando un mes es extremo en CIF pero no en peso, el evento fue de valor y no de volumen: entró una carga '
  'más cara, no más carga. Cuando coinciden ambos, hubo un cambio físico real en la operación.',
  'Estos meses no deben eliminarse. Cada uno debe documentarse en el catálogo de eventos (P32) con su fecha y '
  'su explicación, porque son precisamente los casos que el sistema de alertas tendrá que reconocer.',
  'El z robusto detecta desviaciones respecto de la mediana de <b>toda</b> la serie. Como hay un cambio de nivel '
  'documentado, parte de lo que aparece como extremo puede ser simplemente el régimen nuevo. Conviene repetir '
  'el cálculo por subperiodo.')

In [ ]:
pregunta('P23',
  '¿La variabilidad de la serie cambia con su nivel?',
  'Desviación estándar móvil y coeficiente de variación frente al nivel local.',
  'Serie doble nivel-dispersión y gráfico de dispersión contra nivel.',
  'Decidir si corresponde transformación logarítmica y si el intervalo de predicción debe ser proporcional al nivel en vez de constante.')

nivel = serie['cif_usd'].rolling(12).mean()
disp = serie['cif_usd'].rolling(12).std()
cv = (disp / nivel).dropna()
df_h = pd.concat([nivel.rename('nivel'), disp.rename('disp')], axis=1).dropna()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
a1b = a1.twinx()
a1.plot(nivel.index, nivel / 1e6, color=AZUL, lw=2, label='nivel (media móvil 12)')
a1b.plot(disp.index, disp / 1e6, color=ROJO, lw=1.6, ls='--', label='dispersión (desv. móvil 12)')
a1.set_ylabel('Nivel · millones USD', color=AZUL); a1b.set_ylabel('Dispersión · millones USD', color=ROJO)
a1.set_title('Nivel y dispersión en el tiempo')

a2.scatter(df_h['nivel'] / 1e6, df_h['disp'] / 1e6, s=22, color=GRIS, alpha=.75)
b, a = np.polyfit(df_h['nivel'], df_h['disp'], 1)
xs = np.linspace(df_h['nivel'].min(), df_h['nivel'].max(), 50)
a2.plot(xs / 1e6, (b * xs + a) / 1e6, color=ROJO, lw=2)
a2.set_xlabel('Nivel · millones USD'); a2.set_ylabel('Dispersión · millones USD')
a2.set_title('Dispersión contra nivel')
plt.suptitle('P23 · ¿La variabilidad crece con el nivel?', y=1.02, fontweight='bold')
plt.tight_layout(); plt.show()

corr = df_h['nivel'].corr(df_h['disp'])
crece = corr > 0.3
respuesta(
  f'La correlación entre el nivel y la dispersión móvil es <b>{corr:.3f}</b>. El coeficiente de variación '
  f'oscila entre <b>{cv.min():.3f}</b> y <b>{cv.max():.3f}</b>, con media <b>{cv.mean():.3f}</b>. '
  f'La pendiente ajustada es <b>{b:.4f}</b>.',
  ('La dispersión crece junto con el nivel: la serie es heterocedástica. Los meses de mayor volumen no solo son '
   'más grandes, también son más impredecibles en términos absolutos.' if crece else
   'La dispersión no acompaña claramente al nivel. La varianza se mantiene relativamente estable en términos absolutos.'),
  ('Dos consecuencias directas. La transformación logarítmica del objetivo está justificada, y el pipeline ya la aplica. '
   'Y sobre todo: <b>el intervalo de predicción no debería tener ancho fijo</b>. Actualmente se construye como '
   '± 1,2816 × RMSE, el mismo margen para cualquier nivel. Si la dispersión crece con el nivel, ese intervalo '
   'queda demasiado ancho al principio de la serie y demasiado estrecho al final, que es justo donde se pronostica.'
   if crece else
   'Un intervalo de ancho constante es una aproximación aceptable. Aun así debe verificarse su cobertura empírica (P51).'),
  'La ventana móvil de 12 meses mezcla el efecto de la estacionalidad con el de la dispersión. Una comprobación '
  'más limpia se hace sobre los residuos del modelo frente a los valores ajustados, no sobre la serie cruda.')

---
# Bloque 4 · Dinámica temporal y eventos

In [ ]:
pregunta('P24 · P25',
  '¿Cuál es la evolución mensual del valor CIF y del peso neto?',
  'Serie temporal con media móvil de doce meses.',
  'Líneas superpuestas con su tendencia suavizada.',
  'Identificar niveles, tendencia y momentos de quiebre.')

fig, ejes = plt.subplots(2, 1, figsize=(11.5, 7), sharex=True)
for ax, col, etiq, color in [
        (ejes[0], 'cif_usd', 'CIF · millones USD', AZUL),
        (ejes[1], 'peso_neto_kg', 'Peso neto · millones kg', NARANJA)]:
    ax.plot(serie.index, serie[col], color=GRIS, lw=1, alpha=.75, label='mensual')
    ax.plot(serie.index, serie[col].rolling(12).mean(), color=color, lw=2.4, label='media móvil 12 meses')
    millones(ax); ax.set_ylabel(etiq); ax.legend(frameon=False, fontsize=9)
ejes[0].set_title('P24 y P25 · Evolución mensual de los dos objetivos')
plt.tight_layout(); plt.show()

c_ini, c_fin = serie['cif_usd'].iloc[:12].mean(), serie['cif_usd'].iloc[-12:].mean()
p_ini, p_fin = serie['peso_neto_kg'].iloc[:12].mean(), serie['peso_neto_kg'].iloc[-12:].mean()

respuesta(
  f'El CIF pasó de un promedio de <b>{c_ini/1e6:,.0f} M USD</b> en el primer año a '
  f'<b>{c_fin/1e6:,.0f} M USD</b> en el último, un cambio de <b>{(c_fin/c_ini-1)*100:+.1f} %</b>. '
  f'El peso neto pasó de <b>{p_ini/1e6:,.0f} M kg</b> a <b>{p_fin/1e6:,.0f} M kg</b>, '
  f'<b>{(p_fin/p_ini-1)*100:+.1f} %</b>.',
  'Las dos series crecen, pero no al mismo ritmo. Esa diferencia es exactamente lo que captura el valor '
  'unitario de la P19: si el valor creció más que el peso, la carga se encareció además de aumentar.',
  'Justifica modelar los dos objetivos por separado en vez de derivar uno del otro, que es la decisión de '
  'diseño central del producto.',
  'Comparar el primer año contra el último resume mucho y puede ocultar los quiebres intermedios. '
  'El análisis de cambio estructural (P30) es el que localiza cuándo ocurrieron.')

In [ ]:
pregunta('P28',
  '¿Existe estacionalidad mensual en el CIF, el peso neto y el CIF por kilogramo?',
  'Índices estacionales calculados como razón frente a la media anual móvil.',
  'Diagrama de caja por mes calendario e índice estacional.',
  'Justificar la línea base Naive 12 y las variables de calendario del modelo.')

tmp = serie.copy(); tmp['mes'] = tmp.index.month
indice = tmp.groupby('mes')['cif_usd'].mean() / tmp['cif_usd'].mean()
MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
a1.boxplot([tmp.loc[tmp['mes'] == m, 'cif_usd'] / 1e6 for m in range(1, 13)],
           labels=MESES, patch_artist=True, boxprops=dict(facecolor=AZUL, alpha=.5))
a1.set_ylabel('CIF · millones USD'); a1.set_title('Distribución del CIF por mes')
colores = [VERDE if v >= 1 else NARANJA for v in indice.values]
a2.bar(MESES, indice.values, color=colores, alpha=.85)
a2.axhline(1.0, color=ROJO, ls='--', lw=1.2)
a2.set_ylim(min(indice) * .95, max(indice) * 1.05); a2.set_title('Índice estacional (1,00 = promedio)')
plt.suptitle('P28 · Estacionalidad mensual', y=1.02, fontweight='bold')
plt.tight_layout(); plt.show()

alto, bajo = indice.idxmax(), indice.idxmin()
amplitud = (indice.max() - indice.min()) * 100

respuesta(
  f'El mes más alto es <b>{MESES[alto-1]}</b> con índice <b>{indice.max():.3f}</b> y el más bajo '
  f'<b>{MESES[bajo-1]}</b> con <b>{indice.min():.3f}</b>. La amplitud estacional es de '
  f'<b>{amplitud:.1f} puntos porcentuales</b>.',
  ('Hay un patrón estacional reconocible pero moderado: la diferencia entre el mejor y el peor mes es de un '
   'orden bastante menor que la variación entre años.' if amplitud < 30 else
   'La estacionalidad es marcada y explica una parte apreciable de la variación mensual.'),
  'Justifica incluir las variables de calendario seno y coseno que ya usa el modelo, y sostiene el uso de '
  'Naive 12 como referencia estacional. Pero atención: una amplitud moderada significa que Naive 12 es una '
  'referencia <b>débil</b>, y por eso hay que compararla también contra Naive 1 (ver P47).',
  'El índice se calcula sobre toda la serie, incluyendo el cambio de nivel. Si el patrón estacional cambió '
  'entre regímenes, este promedio lo mezcla. Conviene recalcularlo por subperiodo.')

In [ ]:
pregunta('P29',
  '¿Qué tan persistente es cada indicador respecto de sus rezagos?',
  'Función de autocorrelación simple hasta el rezago 24.',
  'Correlograma con banda de significancia.',
  'Seleccionar qué rezagos entran al modelo como variables.')

def acf(s, k=24):
    s = s.dropna(); s = s - s.mean()
    d = (s ** 2).sum()
    return np.array([1.0] + [(s[j:] * s[:-j]).sum() / d for j in range(1, k + 1)])

MAXLAG = 24
acf_cif, acf_peso = acf(serie['cif_usd'], MAXLAG), acf(serie['peso_neto_kg'], MAXLAG)
banda = 1.96 / np.sqrt(len(serie))

fig, ejes = plt.subplots(1, 2, figsize=(12.5, 4))
for ax, vals, titulo, color in [(ejes[0], acf_cif, 'CIF', AZUL), (ejes[1], acf_peso, 'Peso neto', NARANJA)]:
    ax.bar(range(len(vals)), vals, color=color, alpha=.85, width=.65)
    ax.axhline(banda, color=ROJO, ls='--', lw=1); ax.axhline(-banda, color=ROJO, ls='--', lw=1)
    ax.axhline(0, color='black', lw=.8)
    ax.set_xlabel('Rezago (meses)'); ax.set_title(f'ACF · {titulo}')
plt.suptitle('P29 · Persistencia de los objetivos', y=1.02, fontweight='bold')
plt.tight_layout(); plt.show()

respuesta(
  f'El CIF tiene autocorrelación de <b>{acf_cif[1]:.3f}</b> en el rezago 1 y <b>{acf_cif[12]:.3f}</b> en el 12. '
  f'El peso neto tiene <b>{acf_peso[1]:.3f}</b> y <b>{acf_peso[12]:.3f}</b> respectivamente. '
  f'La banda de significancia está en ±{banda:.3f}.',
  'El CIF es fuertemente persistente: el mes pasado explica buena parte del mes siguiente. El rezago 12 es '
  'bastante más débil que el rezago 1, lo que confirma que la memoria de corto plazo domina sobre la estacional.',
  '<b>Este es el resultado más importante para la evaluación del modelo.</b> Si el rezago 1 es mucho más '
  'informativo que el rezago 12, entonces una línea base que use el mes anterior (Naive 1) será mucho más '
  'difícil de superar que Naive 12. Comparar solo contra Naive 12 sobreestima la calidad del modelo.',
  'La ACF sobre la serie en nivel está inflada por la tendencia: parte de esta persistencia es tendencia, no '
  'memoria genuina. La comprobación rigurosa se hace sobre la serie diferenciada o sin tendencia.')

In [ ]:
pregunta('P30 · P31',
  '¿Existen cambios estructurales, y cómo cambian media, mediana y dispersión entre subperiodos incluyendo 2025 y 2026?',
  'Segmentación por régimen y comparación de estadísticos descriptivos.',
  'Serie con los regímenes sombreados y tabla comparativa.',
  'Definir si la validación usa ventana móvil o expansiva, y cuantificar el cambio de nivel.')

CORTES = ['2012-01-01', '2022-01-01', '2025-01-01', '2027-01-01']
ETIQUETAS = ['2012–2021', '2022–2024', '2025–2026']
serie['regimen'] = pd.cut(serie.index, bins=pd.to_datetime(CORTES), labels=ETIQUETAS, right=False)

fig, ax = plt.subplots(figsize=(11.5, 4.2))
ax.plot(serie.index, serie['cif_usd'], color=GRIS, lw=1, alpha=.7)
ax.plot(serie.index, serie['cif_usd'].rolling(12).mean(), color=AZUL, lw=2.2)
for (ini, fin), etq, c in zip(zip(CORTES[:-1], CORTES[1:]), ETIQUETAS, ['#e8eef7', '#fdf0e3', '#e8f5ee']):
    ax.axvspan(pd.Timestamp(ini), min(pd.Timestamp(fin), serie.index.max()), color=c, zorder=0)
    ax.text(pd.Timestamp(ini), ax.get_ylim()[1] * .97, '  ' + etq, fontsize=9, va='top', color='#40506a')
millones(ax); ax.set_ylabel('CIF · millones USD')
ax.set_title('P30 y P31 · Regímenes de la serie')
plt.tight_layout(); plt.show()

tabla = serie.groupby('regimen')[['cif_usd', 'peso_neto_kg', 'precio_implicito_usd_kg']].agg(['mean', 'median', 'std'])
display((tabla / [1e6, 1e6, 1e6, 1e6, 1e6, 1e6, 1, 1, 1]).round(3))

med = serie.groupby('regimen')['cif_usd'].median()
salto = (med.iloc[1] / med.iloc[0] - 1) * 100 if len(med.dropna()) > 1 else np.nan
ultimo = (med.iloc[2] / med.iloc[1] - 1) * 100 if len(med.dropna()) > 2 else np.nan

respuesta(
  f'La mediana del CIF pasó de <b>{med.iloc[0]/1e6:,.0f} M USD</b> en 2012–2021 a '
  f'<b>{med.iloc[1]/1e6:,.0f} M</b> en 2022–2024, un salto de <b>{salto:+.2f} %</b>. '
  + (f'En 2025–2026 la mediana es <b>{med.iloc[2]/1e6:,.0f} M</b>, un cambio de <b>{ultimo:+.2f} %</b> '
     'respecto del régimen anterior.' if not np.isnan(ultimo) else ''),
  'El salto entre los dos primeros regímenes es grande y no es ruido. Lo relevante es que el periodo más '
  'reciente permite ver si ese nivel nuevo se consolidó o si fue un episodio transitorio, algo que las '
  'versiones anteriores del análisis no comprobaban porque cortaban en 2024.',
  'Si el régimen nuevo se sostiene, una ventana de entrenamiento expansiva arrastra un periodo que ya no '
  'representa la realidad actual y conviene evaluar también una ventana móvil. La comparación de ventanas '
  'de 24, 36 y 48 cortes es justamente lo que resuelve esta duda.',
  'Los cortes de régimen están fijados a mano en enero de 2022 y enero de 2025. Una prueba formal de puntos '
  'de cambio los localizaría a partir de los datos en vez de imponerlos.')

---
# Bloque 5 · Composición y contribución

In [ ]:
# Detecta automáticamente los nombres de columna de los agregados
def detecta(df, claves):
    for c in df.columns:
        if any(k in c.lower() for k in claves):
            return c
    return None

DIM = {}
for nombre, df in [('pais', pais), ('capitulo', capitulo)]:
    if df is None:
        continue
    DIM[nombre] = {
        'df': df,
        'fecha': detecta(df, ['fecha', 'periodo', 'mes']),
        'clave': detecta(df, [nombre[:4], 'cod']),
        'cif': detecta(df, ['cif', 'valor']),
        'peso': detecta(df, ['peso', 'pnk', 'kg']),
    }
    print(nombre, '->', {k: v for k, v in DIM[nombre].items() if k != 'df'})

if not DIM:
    aviso('No se cargaron los agregados por país y capítulo. Las preguntas del Bloque 5 no pueden ejecutarse.')

In [ ]:
pregunta('P33 · P35',
  '¿Qué países y qué capítulos arancelarios concentran el valor CIF?',
  'Participaciones porcentuales sobre el total e índice de concentración Herfindahl-Hirschman.',
  'Pareto de los principales y curva de participación acumulada.',
  'Identificar dependencia comercial y saber qué revisar primero cuando se dispara una alerta.')

def pareto(clave, titulo, color, top=12):
    d = DIM.get(clave)
    if not d or not d['clave'] or not d['cif']:
        aviso(f'No se pudo construir el Pareto de {titulo}: faltan columnas identificables.')
        return None
    tot = d['df'].groupby(d['clave'])[d['cif']].sum().sort_values(ascending=False)
    part = tot / tot.sum() * 100
    hhi = ((tot / tot.sum()) ** 2).sum() * 10000

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
    a1.barh([str(i) for i in part.head(top).index][::-1], part.head(top).values[::-1], color=color, alpha=.85)
    a1.set_xlabel('% del CIF total'); a1.set_title(f'{titulo} · {top} principales')
    a2.plot(range(1, len(part) + 1), part.cumsum().values, color=color, lw=2)
    a2.axhline(80, color=ROJO, ls='--', lw=1, label='80 % del total')
    a2.set_xlabel(f'Número de {titulo.lower()}'); a2.set_ylabel('% acumulado')
    a2.legend(frameon=False, fontsize=9); a2.set_title('Participación acumulada')
    plt.suptitle(f'Concentración por {titulo.lower()}', y=1.02, fontweight='bold')
    plt.tight_layout(); plt.show()
    return part, hhi, tot

res_pais = pareto('pais', 'Países', AZUL)
res_cap = pareto('capitulo', 'Capítulos', NARANJA)

if res_pais and res_cap:
    pp, hhi_p, _ = res_pais
    pc, hhi_c, _ = res_cap
    n80_p = int((pp.cumsum() <= 80).sum()) + 1
    n80_c = int((pc.cumsum() <= 80).sum()) + 1
    respuesta(
      f'Los cinco primeros países concentran <b>{pp.head(5).sum():.1f} %</b> del CIF y bastan '
      f'<b>{n80_p} países</b> para llegar al 80 % del total (HHI = <b>{hhi_p:,.0f}</b>). '
      f'En capítulos, los cinco primeros suman <b>{pc.head(5).sum():.1f} %</b> y hacen falta '
      f'<b>{n80_c} capítulos</b> para el 80 % (HHI = <b>{hhi_c:,.0f}</b>).',
      'La concentración por origen es notablemente mayor que por producto. Unos pocos países explican la mayor '
      'parte del valor, mientras que la canasta de productos está más repartida. Como referencia, un HHI por '
      'encima de 2.500 se considera alta concentración.',
      'El panel descriptivo debe permitir filtrar por estos actores principales. Y cuando el pronóstico dispare '
      'una alerta, la revisión debe empezar por los países del top, que son los que pueden mover el agregado.',
      'Los identificadores son códigos numéricos del DANE, no nombres. <b>Falta incorporar la tabla de '
      'correspondencia oficial</b> para que el dashboard sea legible por un usuario de negocio.')

In [ ]:
pregunta('P36',
  '¿Cómo evoluciona la concentración por país a lo largo del tiempo?',
  'Índice Herfindahl-Hirschman calculado mes a mes.',
  'Serie temporal del índice de concentración.',
  'Evaluar si la canasta de orígenes se diversifica o se concentra.')

d = DIM.get('pais')
if d and d['fecha'] and d['clave'] and d['cif']:
    tmp = d['df'].copy()
    tmp[d['fecha']] = pd.to_datetime(tmp[d['fecha']])
    total_mes = tmp.groupby(d['fecha'])[d['cif']].transform('sum')
    tmp['sh'] = tmp[d['cif']] / total_mes
    hhi_mes = tmp.groupby(d['fecha'])['sh'].apply(lambda s: (s ** 2).sum() * 10000)

    fig, ax = plt.subplots(figsize=(11.5, 4))
    ax.plot(hhi_mes.index, hhi_mes.values, color=GRIS, lw=1, alpha=.8, label='mensual')
    ax.plot(hhi_mes.index, hhi_mes.rolling(12).mean(), color=AZUL, lw=2.4, label='media móvil 12 meses')
    ax.axhline(2500, color=ROJO, ls='--', lw=1, label='umbral de alta concentración')
    ax.set_ylabel('HHI'); ax.legend(frameon=False, fontsize=9)
    ax.set_title('P36 · Evolución de la concentración por país de origen')
    plt.tight_layout(); plt.show()

    ini, fin = hhi_mes.iloc[:12].mean(), hhi_mes.iloc[-12:].mean()
    respuesta(
      f'El HHI por país promedia <b>{ini:,.0f}</b> en el primer año y <b>{fin:,.0f}</b> en el último, '
      f'un cambio de <b>{(fin/ini-1)*100:+.1f} %</b>. El rango histórico va de '
      f'<b>{hhi_mes.min():,.0f}</b> a <b>{hhi_mes.max():,.0f}</b>.',
      ('La concentración aumentó: las importaciones dependen hoy de menos orígenes que al comienzo del periodo.'
       if fin > ini else
       'La concentración disminuyó: la canasta de orígenes está más repartida que al comienzo del periodo.'),
      'La concentración es en sí misma un indicador de riesgo que merece estar en el panel. Cuanto más '
      'concentrada la canasta, más sensible es el agregado a lo que ocurra con un solo origen, y más útil '
      'resulta el pronóstico como herramienta de alerta.',
      'El HHI se calcula sobre valor CIF. La concentración física, medida sobre peso, puede ser distinta: '
      'un país puede pesar mucho en valor y poco en volumen.')
else:
    aviso('No se pudo calcular la evolución del HHI: faltan las columnas de fecha, país o valor.')

---
# Bloque 6 · Variables externas

In [ ]:
pregunta('P40 · P42',
  '¿Qué relación contemporánea y rezagada existe entre la TRM, el ONI y los objetivos?',
  'Correlación cruzada por rezago de 0 a 12 meses sobre las series en variación mensual.',
  'Gráfico de correlación por rezago para cada variable externa.',
  'Justificar la inclusión de cada variable externa como señal predictiva.')

num_ext = ext.select_dtypes(include='number')
base = serie[['cif_usd', 'peso_neto_kg']].join(num_ext, how='inner')
var = base.pct_change().dropna()

MAXR = 12
variables = [c for c in num_ext.columns if c in var.columns][:4]
fig, ejes = plt.subplots(1, len(variables), figsize=(3.4 * len(variables), 3.8), squeeze=False)
resumen_cc = {}
for j, v in enumerate(variables):
    cc = [var['cif_usd'].corr(var[v].shift(k)) for k in range(MAXR + 1)]
    resumen_cc[v] = cc
    ax = ejes[0, j]
    ax.bar(range(MAXR + 1), cc, color=[AZUL if x >= 0 else ROJO for x in cc], alpha=.85)
    ax.axhline(0, color='black', lw=.8)
    ax.axhline(1.96/np.sqrt(len(var)), color=GRIS, ls='--', lw=.9)
    ax.axhline(-1.96/np.sqrt(len(var)), color=GRIS, ls='--', lw=.9)
    ax.set_title(v, fontsize=10); ax.set_xlabel('rezago')
plt.suptitle('P40 y P42 · Correlación cruzada con el CIF (series en variación)', y=1.04, fontweight='bold')
plt.tight_layout(); plt.show()

mejores = {v: (int(np.nanargmax(np.abs(cc))), cc[int(np.nanargmax(np.abs(cc)))]) for v, cc in resumen_cc.items()}
detalle = '; '.join(f'<b>{v}</b>: máximo |r| = {r:.3f} en el rezago {k}' for v, (k, r) in mejores.items())
max_abs = max(abs(r) for _, r in mejores.values())

respuesta(
  f'Correlaciones cruzadas con la variación mensual del CIF — {detalle}. '
  f'La banda de significancia está en ±{1.96/np.sqrt(len(var)):.3f}.',
  ('Ninguna variable externa muestra una correlación fuerte con la variación del CIF. Las magnitudes son '
   'modestas, lo cual es lo esperable: el comercio exterior responde a muchos factores simultáneos.'
   if max_abs < 0.3 else
   'Al menos una variable externa muestra una asociación apreciable con la variación del CIF en algún rezago.'),
  'Una correlación baja no descarta la variable automáticamente, pero sí obliga a demostrar su aporte mediante '
  'análisis de ablación: entrenar con y sin ella y comparar el error fuera de muestra. Si no mejora, sobra, '
  'y quitarla simplifica el modelo sin costo.',
  '<b>Correlación no es causalidad.</b> Estas asociaciones no permiten afirmar que la TRM o el ONI causen '
  'variaciones en las importaciones. Además el cálculo es en muestra: el único criterio válido para conservar '
  'una variable es su aporte fuera de muestra.')

---
# Bloque 7 · Modelado, incertidumbre y utilidad

Este bloque contiene las tres verificaciones que la auditoría señaló como prioritarias: comparar contra la línea base correcta, medir el sesgo y comprobar la cobertura real de los intervalos.

In [ ]:
pregunta('P47',
  '¿Qué desempeño obtienen las líneas base Naive 1, Naive 12 y drift?',
  'Backtest de las tres referencias sobre la misma ventana de 24 meses usada por los modelos.',
  'Tabla comparativa de métricas y serie de errores.',
  'Fijar el mínimo real a superar y establecer el denominador del MASE.')

VENTANA = 24
y = serie['cif_usd'].dropna()
fechas_test = y.index[-VENTANA:]
real = y.reindex(fechas_test)

bases = {
    'Naive 1 (mes anterior)': y.shift(1).reindex(fechas_test),
    'Naive 12 (mismo mes año anterior)': y.shift(12).reindex(fechas_test),
    'Drift (tendencia lineal)': y.shift(1).reindex(fechas_test) + (y.diff().mean()),
}

def met(r, p):
    e = np.abs(r - p)
    return {'MAE': e.mean(), 'RMSE': float(np.sqrt(((r - p) ** 2).mean())),
            'WAPE_pct': e.sum() / np.abs(r).sum() * 100, 'Sesgo': (p - r).mean()}

tabla_b = pd.DataFrame({k: met(real, v) for k, v in bases.items()}).T

if metricas is not None:
    modelo_cif = metricas[metricas['target'] == 'cif_usd'].set_index('modelo')
    for m in modelo_cif.index:
        if 'Naive' not in m:
            tabla_b.loc[f'{m} (modelo del proyecto)'] = {
                'MAE': modelo_cif.loc[m, 'MAE'], 'RMSE': modelo_cif.loc[m, 'RMSE'],
                'WAPE_pct': modelo_cif.loc[m, 'WAPE_pct'], 'Sesgo': np.nan}

mae_naive1 = tabla_b.loc['Naive 1 (mes anterior)', 'MAE']
tabla_b['MASE'] = tabla_b['MAE'] / mae_naive1
tabla_b = tabla_b.sort_values('WAPE_pct')
display(tabla_b.round(3))

fig, ax = plt.subplots(figsize=(11, 4))
colores_b = [VERDE if 'modelo' in i else (ROJO if 'Naive 12' in i else GRIS) for i in tabla_b.index]
ax.barh(tabla_b.index, tabla_b['WAPE_pct'], color=colores_b, alpha=.85)
for i, v in enumerate(tabla_b['WAPE_pct']):
    ax.text(v + .2, i, f'{v:.2f} %', va='center', fontsize=9)
ax.set_xlabel('WAPE (%) — menor es mejor'); ax.set_title('P47 · Líneas base contra los modelos del proyecto')
plt.tight_layout(); plt.show()

w1 = tabla_b.loc['Naive 1 (mes anterior)', 'WAPE_pct']
w12 = tabla_b.loc['Naive 12 (mismo mes año anterior)', 'WAPE_pct']
filas_modelo = [i for i in tabla_b.index if 'modelo' in i]
wm = tabla_b.loc[filas_modelo, 'WAPE_pct'].min() if filas_modelo else np.nan

respuesta(
  f'Sobre los últimos {VENTANA} meses: <b>Naive 1 = {w1:.2f} %</b> de WAPE, '
  f'<b>Naive 12 = {w12:.2f} %</b>'
  + (f' y el mejor modelo del proyecto <b>{wm:.2f} %</b>.' if not np.isnan(wm) else '.')
  + f' Naive 1 es <b>{(1 - w1/w12)*100:.1f} %</b> mejor que Naive 12.',
  'Naive 12 es una referencia mucho más débil que Naive 1, tal como anticipaba la persistencia observada en la '
  'P29. Comparar el modelo únicamente contra Naive 12 exagera su mérito.'
  + (f' Frente a la referencia correcta, la mejora real del modelo es de <b>{(1 - wm/w1)*100:.1f} %</b>, '
     'no la que se obtiene contra la referencia estacional.' if not np.isnan(wm) else ''),
  'La columna MASE está escalada por Naive 1: un valor menor que 1 significa que el modelo aporta sobre la '
  'referencia mínima honesta. <b>Esta tabla es la que debe presentarse en la sustentación</b>, porque responde '
  'por adelantado la pregunta más incómoda que puede hacer un jurado.',
  'Las líneas base se calculan aquí directamente sobre la serie, mientras que las métricas del modelo provienen '
  'del backtest walk-forward guardado. Las ventanas deberían coincidir, pero conviene verificarlo explícitamente '
  'antes de citar la comparación como definitiva.')

In [ ]:
pregunta('P49',
  '¿El modelo presenta sesgo sistemático, error máximo relevante o degradación reciente?',
  'Error con signo por corte, error máximo y desempeño de los cortes más recientes.',
  'Distribución de residuos y serie de error acumulado.',
  'Detectar sobreestimación o subestimación persistente y pérdida de vigencia del modelo.')

if preds is not None:
    for obj in preds['target'].unique():
        sub = preds[preds['target'] == obj]
        mejor = sub.groupby('modelo').apply(lambda g: np.abs(g['real'] - g['prediccion']).sum() / g['real'].sum())
        gan = mejor.idxmin()
        g = sub[sub['modelo'] == gan].sort_values('fecha')
        err = g['prediccion'].values - g['real'].values
        rel = err / g['real'].values * 100

        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 3.8))
        a1.hist(rel, bins=12, color=AZUL, alpha=.8, edgecolor='white')
        a1.axvline(0, color='black', lw=1)
        a1.axvline(rel.mean(), color=ROJO, ls='--', lw=1.6, label=f'sesgo medio {rel.mean():+.2f} %')
        a1.legend(frameon=False, fontsize=9); a1.set_xlabel('Error relativo (%)')
        a1.set_title(f'{obj} · {gan}')
        a2.bar(range(len(rel)), rel, color=[VERDE if x >= 0 else NARANJA for x in rel], alpha=.85)
        a2.axhline(0, color='black', lw=.8)
        a2.set_xlabel('Corte de validación'); a2.set_ylabel('Error relativo (%)')
        a2.set_title('Error por corte')
        plt.tight_layout(); plt.show()

        sobre = int((err > 0).sum())
        recientes = np.abs(rel[-6:]).mean()
        respuesta(
          f'Objetivo <b>{obj}</b>, modelo <b>{gan}</b>: sesgo medio de <b>{rel.mean():+.2f} %</b>, '
          f'error absoluto máximo de <b>{np.abs(rel).max():.2f} %</b>. Sobreestima en '
          f'<b>{sobre} de {len(err)}</b> cortes. El error medio de los últimos 6 cortes es '
          f'<b>{recientes:.2f} %</b> frente a <b>{np.abs(rel).mean():.2f} %</b> del total.',
          ('El sesgo es pequeño y los errores se reparten a ambos lados del cero: el modelo no tira '
           'sistemáticamente hacia arriba ni hacia abajo.' if abs(rel.mean()) < 2 else
           'Hay un sesgo apreciable en una dirección. El modelo se equivoca de forma consistente, no aleatoria.')
          + (' El desempeño reciente es peor que el promedio, lo que sugiere degradación.'
             if recientes > np.abs(rel).mean() * 1.25 else ' El desempeño reciente se mantiene en línea con el promedio.'),
          'El sesgo debe reportarse junto al WAPE. Un modelo con poco error pero sesgado en una dirección '
          'induce decisiones equivocadas de forma sistemática, y eso no lo detecta ninguna métrica de error absoluto.',
          f'Solo hay {len(err)} cortes de validación. Con esa cantidad, el sesgo medio tiene un margen de '
          'incertidumbre amplio y el diagnóstico de degradación sobre los últimos 6 es meramente indicativo.')
else:
    aviso('No se encontró predicciones_validacion.csv. Ejecuta <code>python src/modeling.py</code> primero.')

In [ ]:
pregunta('P51',
  '¿Los intervalos alcanzan la cobertura nominal declarada del 80 %?',
  'Reconstrucción del intervalo con el mismo método del pipeline (± 1,2816 × RMSE) y conteo de cuántas '
  'observaciones reales quedan dentro.',
  'Serie con la banda de predicción y marcadores de los casos fuera.',
  'Calibrar la incertidumbre, o renombrar el intervalo según la cobertura efectivamente medida.')

Z80 = 1.2816
if preds is not None:
    filas = []
    for obj in preds['target'].unique():
        sub = preds[preds['target'] == obj]
        mejor = sub.groupby('modelo').apply(lambda g: np.abs(g['real'] - g['prediccion']).sum() / g['real'].sum())
        gan = mejor.idxmin()
        g = sub[sub['modelo'] == gan].sort_values('fecha').reset_index(drop=True)
        rmse = float(np.sqrt(((g['real'] - g['prediccion']) ** 2).mean()))
        semi = Z80 * rmse
        lo, hi = g['prediccion'] - semi, g['prediccion'] + semi
        dentro = ((g['real'] >= lo) & (g['real'] <= hi))
        cob = dentro.mean() * 100
        filas.append({'Objetivo': obj, 'Modelo': gan, 'RMSE': rmse,
                      'Semiancho': semi, 'Ancho relativo %': semi / g['prediccion'].mean() * 100,
                      'Cobertura nominal %': 80.0, 'Cobertura empírica %': cob,
                      'Fuera': int((~dentro).sum()), 'Cortes': len(g)})

        fig, ax = plt.subplots(figsize=(11.5, 4))
        ax.fill_between(g['fecha'], lo, hi, color=AZUL, alpha=.18, label='intervalo 80 % declarado')
        ax.plot(g['fecha'], g['prediccion'], color=AZUL, lw=1.8, label='predicción')
        ax.plot(g['fecha'], g['real'], color='black', lw=1.6, ls='--', label='real')
        fuera = g[~dentro]
        if len(fuera):
            ax.scatter(fuera['fecha'], fuera['real'], color=ROJO, s=70, zorder=5,
                       label=f'{len(fuera)} fuera del intervalo')
        millones(ax); ax.legend(frameon=False, fontsize=9)
        ax.set_title(f'P51 · Cobertura del intervalo — {obj} ({gan}) — cobertura real {cob:.1f} %')
        plt.tight_layout(); plt.show()

    tabla_cob = pd.DataFrame(filas)
    display(tabla_cob.round(2))

    cobs = tabla_cob['Cobertura empírica %']
    bajo = (cobs < 75).any()
    respuesta(
      'Cobertura empírica medida sobre los cortes del backtest: '
      + '; '.join(f'<b>{r.Objetivo}</b> {r._8:.1f} % frente al 80 % declarado ({r.Fuera} de {r.Cortes} fuera)'
                  for r in tabla_cob.itertuples()) + '.',
      ('Al menos un intervalo cubre menos de lo que declara: promete más precisión de la que el modelo tiene.'
       if bajo else
       'Los intervalos cubren aproximadamente lo que declaran. El método basado en el RMSE del backtest resulta '
       'razonable para esta serie.'),
      'Este número debe aparecer en el documento y en el panel de calidad, junto al intervalo. '
      'Si la cobertura no alcanza el nivel nominal, hay dos salidas honestas: recalibrar el ancho, o renombrar '
      'el intervalo con la cobertura realmente medida. Lo que no es admisible es seguir llamándolo 80 % sin comprobarlo.',
      '<b>La cobertura está medida sobre los mismos cortes que se usaron para elegir el modelo y para estimar '
      'el RMSE.</b> Es por tanto una estimación optimista: la cobertura real sobre datos nuevos será menor. '
      'La verificación limpia requiere un conjunto de prueba que no haya participado en la selección.')
else:
    aviso('No se encontró predicciones_validacion.csv. Ejecuta <code>python src/modeling.py</code> primero.')

---
# Preguntas que requieren los microdatos completos

Las siguientes preguntas operan sobre el registro individual y no pueden responderse desde los archivos consolidados del repositorio. Necesitan los 8,3 GB de `data/raw/` y deben ejecutarse en la máquina local.

| ID | Pregunta | Qué necesita |
|---|---|---|
| P01–P04 | Manifiesto de fuentes, conteos por archivo, esquemas y tipos por vigencia | Los 18 paquetes ZIP del DANE |
| P06 | Duplicados en las capas raw, landing y trusted | Las tres capas completas |
| P07 | Ceros iniciales en los códigos de aduana, país y subpartida | Los microdatos sin transformar |
| P08 | Bitácora de registros excluidos y su razón | Ejecutar el pipeline con registro de exclusiones |
| P10 | Valores negativos, imposibles o fuera de dominio | Nivel de registro |
| P12 | Coherencia entre peso bruto y peso neto | El peso bruto no está en la serie agregada |
| P13–P15 | Unidades por vigencia, cambios metodológicos, códigos no identificados | Microdatos y documentación oficial del DANE |
| P16 | Reconciliación contra los totales oficiales | Boletines técnicos del DANE del periodo |
| P20 | Proporción del total mensual que aportan pocos registros | Nivel de registro |
| P43–P44 | Calendario de disponibilidad y revisiones posteriores del DANE | Descargas fechadas en momentos distintos |
| P45–P46 | Ablación y multicolinealidad | Reejecutar el backtest por conjunto de variables |
| P52 | Reglas de alerta | Definición de umbrales con el usuario |

---

## Cómo se usa esto en el documento académico

Cada respuesta de este notebook está separada en cuatro partes —evidencia, interpretación, implicación y limitación— porque es la estructura que exige el EDA. Al redactar el informe, la evidencia va a la sección de resultados, la implicación a la de metodología o modelado, y la limitación a la de limitaciones. Así ninguna afirmación del documento queda sin respaldo.

Las figuras se pueden exportar con `plt.savefig()` dentro de cada celda si se necesitan como archivos independientes.

> **Regla que no debe romperse:** ninguna cifra puede escribirse a mano en el documento. Toda cifra debe provenir de una celda ejecutada o de un archivo generado por el pipeline.